<a href="https://colab.research.google.com/github/georgmandryka-cyber/lager-chatbot-rag/blob/main/lager_chatbot_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install -q anthropic sentence-transformers

import anthropic
from google.colab import userdata
from sentence_transformers import SentenceTransformer, util

# 1. Wissen
chunks = [
    {"thema": "Gefahrgut", "quelle": "gefahrgut.txt",
     "text": "Gefahrgut wird nur im gekennzeichneten Bereich C gelagert."},
    {"thema": "Gefahrgut", "quelle": "gefahrgut.txt",
     "text": "Entzündbare Stoffe dürfen nicht über 2 Paletten hoch gestapelt werden."},
    {"thema": "Gefahrgut", "quelle": "gefahrgut.txt",
     "text": "Bei einem Leck wird sofort der Bereich abgesperrt und der Schichtleiter informiert."},
    {"thema": "Stapler", "quelle": "stapler.txt",
     "text": "Vor jeder Schicht prüft der Fahrer Gabeln, Bremsen und Hupe."},
    {"thema": "Stapler", "quelle": "stapler.txt",
     "text": "Mängel werden sofort dem Schichtleiter gemeldet, der Stapler wird gesperrt."},
    {"thema": "Wareneingang", "quelle": "wareneingang.txt",
     "text": "Bei Abweichungen von mehr als 2 Prozent wird der Schichtleiter informiert."},
    {"thema": "Verkehr", "quelle": "verkehr.txt",
     "text": "Die Höchstgeschwindigkeit in der Halle beträgt 10 km/h."},
]

# 2. Retrieval mit Embeddings (Thema + Text)
modell = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
chunk_vektoren = modell.encode([c["thema"] + ": " + c["text"] for c in chunks])
SCHWELLE = 0.35

def suche(frage, n=3):
    frage_vektor = modell.encode(frage)
    werte = util.cos_sim(frage_vektor, chunk_vektoren)[0].tolist()
    treffer = sorted(zip(werte, chunks), key=lambda t: t[0], reverse=True)
    return treffer[:n]

# 3. Generation mit Claude
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

SYSTEM = (
    "Du bist ein Assistent für Lager-Arbeitsanweisungen. "
    "Antworte kurz und auf Deutsch, nur anhand der gegebenen Auszüge. "
    "Steht die Antwort nicht in den Auszügen, sag das ehrlich. "
    "Nenne am Ende die Quelle."
)

def antworten(frage):
    treffer = suche(frage)
    if treffer[0][0] < SCHWELLE:
        return "Dazu habe ich nichts in den Unterlagen gefunden."
    kontext = "\n".join(f"- ({c['quelle']}) {c['text']}" for _, c in treffer)
    antwort = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=300,
        system=SYSTEM,
        messages=[{"role": "user",
                   "content": f"Auszüge:\n{kontext}\n\nFrage: {frage}"}],
    )
    return antwort.content[0].text

# 4. Test
fragen = [
    "Wo lagere ich Gefahrgut?",
    "Wo wird Gefahrgut gelagert?",
    "Wie schnell darf der Stapler fahren?",
    "Was passiert bei einem Leck?",
    "Wer ist der Bundeskanzler?",
    "Wie ist das Wetter heute?",
]

for frage in fragen:
    bester = suche(frage, 1)[0][0]
    print(f"FRAGE: {frage}  (bester Wert: {bester:.2f})")
    print(antworten(frage))
    print()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

FRAGE: Wo lagere ich Gefahrgut?  (bester Wert: 0.42)
Gefahrgut wird ausschließlich im **gekennzeichneten Bereich C** gelagert.

*Quelle: gefahrgut.txt*

FRAGE: Wo wird Gefahrgut gelagert?  (bester Wert: 0.58)
Gefahrgut wird im **gekennzeichneten Bereich C** gelagert.

*Quelle: gefahrgut.txt*

FRAGE: Wie schnell darf der Stapler fahren?  (bester Wert: 0.60)
Die Höchstgeschwindigkeit in der Halle beträgt **10 km/h**.

📄 *Quelle: verkehr.txt*

FRAGE: Was passiert bei einem Leck?  (bester Wert: 0.46)
Bei einem Leck wird sofort der Bereich abgesperrt und der Schichtleiter informiert.

**Quelle:** gefahrgut.txt

FRAGE: Wer ist der Bundeskanzler?  (bester Wert: 0.24)
Dazu habe ich nichts in den Unterlagen gefunden.

FRAGE: Wie ist das Wetter heute?  (bester Wert: 0.25)
Dazu habe ich nichts in den Unterlagen gefunden.

